# Optimized+augmented PyTorch CNN

In this notebook, we train the winning CNN architecture from the Optuna run in notebook 04 on the CIFAR-10 dataset with image augmentation for improved generalization.

## Notebook set-up

### Imports

In [ ]:
# Standard library imports
import pickle
# from pathlib import Path

# Third party imports
import matplotlib.pyplot as plt
import numpy as np
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms

# Package imports
import image_classification_tools.pytorch.data as data_utils
import image_classification_tools.pytorch.evaluation as eval_utils
import image_classification_tools.pytorch.hyperparameter_optimization as optimization
import image_classification_tools.pytorch.plotting as plots
import image_classification_tools.pytorch.training as training

# Local imports
import configuration as config


### Configuration

### Hyperparameters

In [ ]:
# Split sizes
val_size = 10000

# Training
epochs = 100
print_every = 10

## 1. Load optimization results

In [ ]:
# Get optimization results from disk
study = optuna.load_study(
    study_name='cnn_optimization',
    storage=config.OPTUNA_STORAGE_URL
)

# Extract hyperparameters from winning trial
best_params = study.best_trial.params
batch_size = best_params['batch_size']

print('Loaded best hyperparameters from Optuna study:')

for key, value in best_params.items():
    print(f'  {key}: {value}')

print(f'\nBest validation accuracy from optimization: {study.best_trial.value:.2f}%')


## 2. Load and preprocess CIFAR-10 data

Load the images into PyTorch tensors and split them into training, validation and testing datasets. Augmentation is applied on-the-fly during training for better generalization.

### 2.1. Define preprocessing transforms

In [ ]:
# Training transform with on-the-fly augmentation
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.5),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))], p=0.1),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
])

# Evaluation transform (no augmentation)
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

### 2.2. Load the training and testing datasets from disk

In [ ]:
# Training data with augmentation
train_dataset = data_utils.load_dataset(
    data_source=datasets.CIFAR10,
    transform=train_transform,
    root=config.DATA_DIR
)

# Test data without augmentation
test_dataset = data_utils.load_dataset(
    data_source=datasets.CIFAR10,
    transform=eval_transform,
    root=config.DATA_DIR,
    train=False
)

### 2.3. Training, validation and testing split

In [ ]:
train_dataset, val_dataset, test_dataset = data_utils.prepare_splits(
    train_dataset=train_dataset,
    test_dataset=test_dataset,
    val_size=val_size
)

### 2.4. Create dataloaders

Create DataLoaders with on-the-fly augmentation applied during training.

In [ ]:
# Create DataLoaders (augmentation applied on-the-fly during training)
train_loader, val_loader, test_loader = data_utils.create_dataloaders(
    train_dataset, val_dataset, test_dataset,
    batch_size=batch_size,
    preload_to_memory=False,  # Keep False for on-the-fly augmentation
    device=config.DEVICE
)

print(f'Training set: {len(train_dataset)} images')
print(f'Validation set: {len(val_dataset)} images')
print(f'Test set: {len(test_dataset)} images')

## 3. Build optimized CNN using best hyperparameters

Create a CNN using the best hyperparameters found during Optuna optimization, then train it with on-the-fly data augmentation.

### 3.1. Create model with best hyperparameters

In [ ]:
# Create model with best hyperparameters from Optuna
model = optimization.create_cnn(
    n_conv_blocks=best_params['n_conv_blocks'],
    initial_filters=best_params['initial_filters'],
    n_fc_layers=best_params['n_fc_layers'],
    conv_dropout_rate=best_params['conv_dropout_rate'],
    fc_dropout_rate=best_params['fc_dropout_rate'],
    num_classes=len(config.CLASS_NAMES),
    in_channels=3
).to(config.DEVICE)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model)
print(f'\nTotal parameters: {trainable_params:,}')


### 3.2. Define loss function and optimizer

In [ ]:
criterion = nn.CrossEntropyLoss()

# Create optimizer with best learning rate (fixed Adam optimizer)
optimizer = optim.Adam(
    model.parameters(), 
    lr=best_params['learning_rate']
)

print('Optimizer: Adam')

### 3.3. Train model

In [ ]:
%%time

history = training.train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=config.DEVICE,
    lazy_loading=True,  # Move data to device during training (not preloaded)
    epochs=epochs,
    print_every=print_every,
    enable_early_stopping=True,
    early_stopping_patience=10
)

print()

### 3.4. Learning curves

In [ ]:
fig, axes = plots.plot_learning_curves(history)
plt.show()

## 4. Evaluate model on test set

### 4.1. Calculate test accuracy

In [ ]:
test_accuracy, predictions, true_labels = eval_utils.evaluate_model(model, test_loader)
print(f'Test accuracy: {test_accuracy:.2f}%')

### 4.2. Per-class accuracy

In [ ]:
# Calculate per-class accuracy
class_correct = {name: 0 for name in config.CLASS_NAMES}
class_total = {name: 0 for name in config.CLASS_NAMES}

for pred, true in zip(predictions, true_labels):

    class_name = config.CLASS_NAMES[true]
    class_total[class_name] += 1

    if pred == true:
        class_correct[class_name] += 1

print('Per-class accuracy:')
print('-' * 30)

for name in config.CLASS_NAMES:
    acc = 100 * class_correct[name] / class_total[name]
    print(f'{name:12s}: {acc:.2f}%')

### 4.3. Confusion matrix

In [ ]:
fig, ax = plots.plot_confusion_matrix(true_labels, predictions, config.CLASS_NAMES)
plt.show()

### 4.4. Predicted class probability distributions

In [ ]:
# Get predicted probabilities for all test samples
model.eval()
all_probs = []

with torch.no_grad():
    for images, _ in test_loader:
        images = images.to(config.DEVICE, non_blocking=True)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        all_probs.append(probs.cpu().numpy())

all_probs = np.concatenate(all_probs, axis=0)

# Plot probability distributions
fig, axes = plots.plot_class_probability_distributions(all_probs, config.CLASS_NAMES)
plt.show()

### 4.5. Evaluation curves

In [ ]:
fig, (ax1, ax2) = plots.plot_evaluation_curves(true_labels, all_probs, config.CLASS_NAMES)
plt.show()

## 5. Save model

In [ ]:
# Save trained model
model_path = config.MODELS_DIR / 'augmented_cnn.pth'
torch.save(model, model_path)

print(f'Model saved to: {model_path}')
print(f'Test accuracy: {test_accuracy:.2f}%')

## 6. Save test results for comparison

In [ ]:
# Save results path
results_path = config.RESULTS_DIR / 'augmented_cnn_results.pkl'

# Count model parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# Create results dictionary
results_dict = {
    'true_labels': true_labels,
    'predictions': predictions,
    'all_probs': all_probs,
    'test_accuracy': test_accuracy,
    'total_params': total_params,
    'trainable_params': trainable_params
}

# Save results
with open(results_path, 'wb') as f:
    pickle.dump(results_dict, f)

print(f'Test results saved to: {results_path}')
print(f'  - Test accuracy: {test_accuracy:.2f}%')
print(f'  - Total parameters: {total_params:,}')
print(f'  - Trainable parameters: {trainable_params:,}')
